# LoRA vs QLoRA, Head-to-Head, on One Generative LLM

**NYP School of Information Technology — Applied AI Series**

Companion notebook to `production_nlp_peft_lora_qlora.ipynb` and `Gemma270m_finetune.ipynb`.
Those two each answered half the question:

- `production_nlp_peft_lora_qlora.ipynb` compares **Full FT vs LoRA vs QLoRA**, but on
  `bert-base-uncased` — an *encoder* used for classification, where the notebook's own
  pedagogical note admits 110M parameters is too small for QLoRA's memory savings to show up.
- `Gemma270m_finetune.ipynb` fine-tunes a real *decoder* LLM, but only demonstrates QLoRA
  (via Unsloth) — there's no plain-LoRA run on the same model to compare against.

This notebook closes that gap: **the same generative decoder LLM
(`TinyLlama/TinyLlama-1.1B-Chat-v1.0`), fine-tuned twice** — once with plain LoRA (fp16
base weights) and once with QLoRA (4-bit NF4 base weights) — using the vanilla
`transformers` + `peft` + `bitsandbytes` stack (no Unsloth), so every step is visible.

**What you will do:**

1. Load `TinyLlama-1.1B-Chat-v1.0` and the `mlabonne/guanaco-llama2-1k` instruction dataset
2. Fine-tune it with **LoRA** (fp16 backbone) and measure trainable params / peak VRAM / time
3. Fine-tune the *same* model with **QLoRA** (4-bit NF4 backbone) and measure the same
4. Compare the two head-to-head in a table and charts
5. Reload each adapter fresh and compare generations qualitatively

**Designed to run on the free Colab GPU (T4, ~15GB VRAM).**
Go to `Runtime → Change runtime type → T4 GPU` before you start.

> **Pedagogical note:** at 1.1B parameters, this model already fits on a T4 in plain fp16
> LoRA — so don't expect QLoRA to look like a *requirement* here. What you're measuring is
> the VRAM headroom QLoRA buys you even when you don't strictly need it (room for bigger
> batches / longer sequences), while confirming the two approaches land at similar loss.
> The scenario where QLoRA stops being optional — fitting a 7B+ model that plain fp16
> simply cannot fit on a T4 at all — is called out as an extension exercise at the end.


## 0. Environment Setup

Same production stack as `production_nlp_peft_lora_qlora.ipynb`: `transformers`, `datasets`,
`accelerate`, `peft`, `bitsandbytes`. No `trl`/`unsloth` here — we tokenize and train with
plain `Trainer`, so every step of turning `guanaco-llama2-1k`'s `text` field into a causal-LM
training batch is explicit rather than hidden behind a wrapper.


In [ ]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes
# Colab sometimes ships an old torchao that's incompatible with this peft version and
# makes get_peft_model() raise ImportError. We don't use torchao here, so remove it.
!pip uninstall -y -q torchao

In [ ]:
import os
import time
import gc
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType, prepare_model_for_kbit_training

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

assert torch.cuda.is_available(), "No GPU detected — go to Runtime > Change runtime type > T4 GPU"
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("Total VRAM: {:.1f} GB".format(torch.cuda.get_device_properties(0).total_memory / 1e9))
print("bf16 supported:", torch.cuda.is_bf16_supported())  # False on T4 (Turing) — we use fp16 instead

## 1. Model & Dataset

`TinyLlama/TinyLlama-1.1B-Chat-v1.0` — a 1.1B-parameter Llama-architecture chat model,
ungated, ~2.2GB in fp16. Same architecture family (and same `target_modules` naming:
`q_proj`/`k_proj`/`v_proj`/`o_proj`/`gate_proj`/`up_proj`/`down_proj`) as the 270M Gemma
model and the 7B Llama/Mistral models you'd meet in production — just big enough to make
LoRA-vs-QLoRA memory numbers meaningfully different from noise.

We reuse [`mlabonne/guanaco-llama2-1k`](https://huggingface.co/datasets/mlabonne/guanaco-llama2-1k)
for consistency with `Gemma270m_finetune.ipynb` — 1,000 pre-formatted instruction/response
examples, split 90/10 here so we have a held-out `eval_loss` to compare strategies on.

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Llama tokenizers ship with no pad token by default

In [ ]:
raw = load_dataset("mlabonne/guanaco-llama2-1k", split="train")
split = raw.train_test_split(test_size=0.1, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]

print(f"Train examples: {len(train_ds)} | Eval examples: {len(eval_ds)}")
print(train_ds[0]["text"][:400])

In [ ]:
def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
eval_tok = eval_ds.map(tokenize_fn, batched=True, remove_columns=eval_ds.column_names)

# mlm=False -> causal LM collator: labels are just a shifted copy of input_ids, and padding
# tokens are masked out of the loss automatically.
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## 2. Benchmark Harness

Same pattern as `production_nlp_peft_lora_qlora.ipynb`'s `run_benchmark`: one shared
function that trains whatever `peft`-wrapped model you hand it, and returns trainable
params, peak GPU memory, wall-clock time, eval loss, and perplexity (`exp(eval_loss)`) —
our stand-in for classification F1 on a generative task, since lower held-out loss means
the model got better at predicting the instruction-tuned responses.

In [ ]:
results = []  # one dict per strategy, for the final comparison table

def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

def run_benchmark(strategy_name, model, run_name, is_quantized, num_epochs=1,
                   lr=2e-4, batch_size=4, grad_accum=2):
    '''Trains `model` with the Trainer API, tracks peak memory/time, returns a metrics dict.'''
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

    trainable, total = count_trainable_params(model)
    print(f"[{strategy_name}] trainable params: {trainable:,} / {total:,} "
          f"({100 * trainable / total:.3f}%)")

    training_args = TrainingArguments(
        output_dir=f"./outputs/{run_name}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        learning_rate=lr,
        warmup_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=20,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        # Paged optimizer avoids memory spikes from 4-bit training, per the QLoRA paper.
        optim="paged_adamw_8bit" if is_quantized else "adamw_torch",
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=eval_tok,
        data_collator=data_collator,
    )

    start = time.time()
    trainer.train()
    train_time = time.time() - start

    eval_metrics = trainer.evaluate()
    eval_loss = eval_metrics["eval_loss"]
    peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9

    record = {
        "strategy": strategy_name,
        "trainable_params": trainable,
        "total_params": total,
        "pct_trainable": 100 * trainable / total,
        "peak_gpu_mem_gb": round(peak_mem_gb, 3),
        "train_time_sec": round(train_time, 1),
        "eval_loss": round(eval_loss, 4),
        "perplexity": round(float(np.exp(eval_loss)), 2),
    }
    results.append(record)

    del trainer
    torch.cuda.empty_cache()
    gc.collect()
    return record

## 3. Approach A — LoRA (fp16 backbone)

Load the backbone at full fp16 precision (no quantization) and freeze it, then attach
trainable low-rank adapters to every linear projection in each decoder block — the
attention `q/k/v/o` projections *and* the gated MLP (`gate_proj`/`up_proj`/`down_proj`),
which is the standard target-module set for Llama-family models.

In [ ]:
ADAPTER_DIR_LORA = "./outputs/tinyllama-lora-adapter"

lora_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
).to(device)
lora_base.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

lora_model = get_peft_model(lora_base, lora_config)
lora_model.print_trainable_parameters()

In [ ]:
lora_results = run_benchmark(
    strategy_name="LoRA (fp16, r=16)",
    model=lora_model,
    run_name="lora-r16",
    is_quantized=False,
)

lora_model.save_pretrained(ADAPTER_DIR_LORA)
tokenizer.save_pretrained(ADAPTER_DIR_LORA)
print(f"Adapter saved to {ADAPTER_DIR_LORA}")

del lora_model, lora_base
torch.cuda.empty_cache()
gc.collect()
lora_results

## 4. Approach B — QLoRA (4-bit NF4 backbone)

Exact same recipe as `production_nlp_peft_lora_qlora.ipynb` Section 6, just applied to a
causal LM instead of a sequence classifier: quantize the frozen backbone to 4-bit NF4 with
`BitsAndBytesConfig`, run `prepare_model_for_kbit_training` (casts norms to fp32, enables
gradient checkpointing, etc.), then attach LoRA adapters exactly as in Approach A.

In [ ]:
ADAPTER_DIR_QLORA = "./outputs/tinyllama-qlora-adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 (Turing) has no bf16 tensor cores
    bnb_4bit_use_double_quant=True,
)

qlora_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
)
qlora_base.config.pad_token_id = tokenizer.pad_token_id
qlora_base = prepare_model_for_kbit_training(qlora_base)

qlora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

qlora_model = get_peft_model(qlora_base, qlora_config)
qlora_model.print_trainable_parameters()

In [ ]:
qlora_results = run_benchmark(
    strategy_name="QLoRA (4-bit, r=16)",
    model=qlora_model,
    run_name="qlora-r16",
    is_quantized=True,
)

qlora_model.save_pretrained(ADAPTER_DIR_QLORA)
tokenizer.save_pretrained(ADAPTER_DIR_QLORA)
print(f"Adapter saved to {ADAPTER_DIR_QLORA}")

del qlora_model, qlora_base
torch.cuda.empty_cache()
gc.collect()
qlora_results

> If this cell errors with a `bitsandbytes` / CUDA mismatch, restart the runtime
> (`Runtime → Restart session`) and re-run from the top — this is a common one-time issue
> after installing `bitsandbytes` fresh into a Colab environment.

## 5. Comparison: LoRA vs QLoRA

Same table/chart pattern as `production_nlp_peft_lora_qlora.ipynb` Section 7, now with two
rows instead of three (no full fine-tuning baseline here — at 1.1B params that would be
the slow, high-memory outlier we already established in the BERT notebook).

In [ ]:
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df[[
    "strategy", "trainable_params", "pct_trainable",
    "peak_gpu_mem_gb", "train_time_sec", "eval_loss", "perplexity"
]]
comparison_df.columns = [
    "Strategy", "Trainable Params", "% Trainable",
    "Peak GPU Mem (GB)", "Train Time (s)", "Eval Loss", "Perplexity"
]
comparison_df

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(comparison_df["Strategy"], comparison_df["Peak GPU Mem (GB)"], color="#1C7293")
axes[0].axhline(16, color="red", linestyle="--", linewidth=1, label="T4 VRAM (16GB)")
axes[0].set_title("Peak GPU Memory (GB)")
axes[0].legend(fontsize=8)
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(comparison_df["Strategy"], comparison_df["Train Time (s)"], color="#0F3B5C")
axes[1].set_title("Training Time (s)")
axes[1].tick_params(axis="x", rotation=20)

axes[2].bar(comparison_df["Strategy"], comparison_df["Perplexity"], color="#4FD1C5")
axes[2].set_title("Eval Perplexity (lower = better)")
axes[2].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig("./outputs/lora_vs_qlora_comparison.png", dpi=150)
plt.show()

## 6. Qualitative Check

Numbers aside — do the two fine-tunes actually *sound* similar? Reload each adapter fresh
on top of a clean copy of the base model (same pattern as `Gemma270m_finetune.ipynb`
Section 5, so training-time state can't leak into the comparison), and generate on the same
held-out prompts.

In [ ]:
test_prompts = [
    "[INST] Explain the concept of machine learning in simple terms. [/INST]",
    "[INST] What are the benefits of using Python for data science? [/INST]",
    "[INST] How does a neural network learn from data? [/INST]",
]

def generate(model, tok, prompts, max_new_tokens=100, temperature=0.7):
    responses = []
    model.eval()
    for prompt in prompts:
        inputs = tok(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
                pad_token_id=tok.pad_token_id,
            )
        responses.append(tok.decode(outputs[0], skip_special_tokens=True))
    return responses

In [ ]:
# Fresh fp16 base + LoRA adapter
base_for_lora = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(device)
lora_infer_model = PeftModel.from_pretrained(base_for_lora, ADAPTER_DIR_LORA)

lora_generations = generate(lora_infer_model, tokenizer, test_prompts)

del lora_infer_model, base_for_lora
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# Fresh 4-bit base + QLoRA adapter
base_for_qlora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map={"": 0}
)
qlora_infer_model = PeftModel.from_pretrained(base_for_qlora, ADAPTER_DIR_QLORA)

qlora_generations = generate(qlora_infer_model, tokenizer, test_prompts)

del qlora_infer_model, base_for_qlora
torch.cuda.empty_cache()
gc.collect()

In [ ]:
for prompt, lora_resp, qlora_resp in zip(test_prompts, lora_generations, qlora_generations):
    print("PROMPT:", prompt)
    print("-" * 80)
    print("LoRA  :", lora_resp)
    print("-" * 80)
    print("QLoRA :", qlora_resp)
    print("=" * 80, "\n")

## 7. Discussion & Extension

**Discussion prompts:**

- How close are `Eval Loss` / `Perplexity` between the two strategies? Given QLoRA trains
  the *same rank* adapters on a *quantized* backbone, would you expect it to ever beat
  plain LoRA on quality — or only approach it while using less memory?
- Look at the `Peak GPU Memory` chart against the 16GB T4 line. At 1.1B parameters, both
  strategies have huge headroom. Which lever would you pull first if you needed to train a
  *larger* batch size or *longer* sequences within that same headroom — increasing LoRA
  rank, or switching to QLoRA to free up more room?
- `production_nlp_peft_lora_qlora.ipynb` made the same "too small to matter" caveat about
  `bert-base-uncased` (110M params). Has the memory gap between LoRA and QLoRA grown,
  shrunk, or stayed about the same going from 110M → 1.1B? Extrapolate: what do you expect
  at 7B?

**Extension exercise — where QLoRA stops being optional:**

Swap `MODEL_NAME` for an ungated 7B model (e.g. `"NousResearch/Llama-2-7b-hf"` or
`"mistralai/Mistral-7B-v0.1"`) and re-run just the QLoRA section (Section 4). A 7B model in
plain fp16 needs ~14GB of VRAM for weights *alone*, before gradients/optimizer states/
activations — Approach A (Section 3) will OOM on a T4 at that size. QLoRA's 4-bit backbone
shrinks that to ~4-5GB, which is the scenario the memory savings actually become the
difference between "trains on a free Colab GPU" and "doesn't fit at all."

**Checklist before you submit:**

- [ ] Both strategies (LoRA, QLoRA) ran successfully and appear in `comparison_df`
- [ ] Comparison chart saved to `./outputs/lora_vs_qlora_comparison.png`
- [ ] Both adapters saved (`./outputs/tinyllama-lora-adapter`, `./outputs/tinyllama-qlora-adapter`)
- [ ] Short written answer (3-5 sentences) to the discussion prompts above, referencing your
      actual peak-memory and perplexity numbers
